### Test `refwrangle.merge_pdfs_with_structure(pdfs_info, output_path)`

Merge pdfs into a single pdf, intended to have RAG-friendly page and bookmark structure

In [2]:
import pathlib as pl
import sys
import io
from pypdf import PdfWriter, PdfReader
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import letter

refwrangle_dir = pl.Path('~/ref/refwrangle').expanduser() # can't reliably get dir of your .ipynb 
sys.path.append(str(refwrangle_dir))
import refwrangle as rfw

def create_sample_pdf(filename, title, bookmarks=None):
    packet = io.BytesIO()
    c = canvas.Canvas(packet, pagesize=letter)
    
    # Main content
    c.setFont("Helvetica", 12)
    c.drawString(100, 750, title)
    c.showPage()
    c.save()
    packet.seek(0)

    new_pdf = PdfReader(packet)
    writer = PdfWriter()
    writer.add_page(new_pdf.pages[0])

    # Add bookmarks only if provided
    if bookmarks:
        for bookmark_title, page in bookmarks:
            writer.add_outline_item(bookmark_title, page)

    with open(filename, 'wb') as f:
        writer.write(f)

def create_unstructured_pdf(filename, title):
    packet = io.BytesIO()
    c = canvas.Canvas(packet, pagesize=letter)
    c.setFont("Helvetica", 12)
    c.drawString(100, 750, title)
    c.showPage()
    c.save()
    packet.seek(0)

    new_pdf = PdfReader(packet)
    writer = PdfWriter()
    writer.add_page(new_pdf.pages[0])

    with open(filename, 'wb') as f:
        writer.write(f)

def verify_bookmarks(pdf_path):
    reader = PdfReader(pdf_path)
    
    def print_bookmark_tree(bookmarks, level=0):
        for item in bookmarks:
            if isinstance(item, list):
                print_bookmark_tree(item, level + 1)
            else:
                indent = "  " * level
                page_num = reader.get_destination_page_number(item)
                print(f"{indent}- {item.title} (Page {page_num})")
    
    print("\nBookmark structure:")
    print_bookmark_tree(reader.outline)

def run_tests():
    print("Creating sample PDFs...")
    
    # Create structured PDFs
    create_sample_pdf("tmp_test_article1.pdf", "Article 1", [
        ("Section 1.1", 0),
        ("Section 1.2", 0)
    ])
    
    create_sample_pdf("tmp_test_article2.pdf", "Article 2", [
        ("Section 2.1", 0),
        ("Section 2.2", 0)
    ])
    
    # Create an unstructured PDF
    create_unstructured_pdf("tmp_test_article3.pdf", "Article 3 - No Structure")

    sample_metadata = [
    {
        "Title_1": "First Research Paper",
        "Author": "John Smith",
        "Source": "Science Journal",
        "Date": "2024-01-15"
    },
    {
        "Title_2": "Second Research Paper",
        "Author": "Jane Doe",
        "Source": "Nature",
        "Date": "2024-02-20",
        'Extra Key': 'exra'
    },
    {
        "Title_3": "Third Research Paper",
        "Author": "Bob Johnson",
        "Source": "Research Quarterly",
        "Date": "2024-03-10"
    } ]
    pdfs_info = [{'file': f'tmp_test_article{i+1}.pdf', 'metainfo': sample_metadata[i]} for i in range(3)]
    
    print("Merging PDFs...")
    #pdf_files = ["article1.pdf", "article2.pdf", "article3.pdf"]
    rfw.merge_pdfs_with_structure(pdfs_info, "merged_articles.pdf")
    
    print("Verifying merged PDF structure...")
    verify_bookmarks("merged_articles.pdf")
    
    print("\nTest completed. Please check merged_articles.pdf")

if __name__ == "__main__":
    run_tests()

Creating sample PDFs...
Merging PDFs...
Verifying merged PDF structure...

Bookmark structure:
- Article 1: tmp_test_article1 (Page 1)
  - Section 1.1 (Page 2)
  - Section 1.2 (Page 2)
- Article 2: tmp_test_article2 (Page 3)
  - Section 2.1 (Page 4)
  - Section 2.2 (Page 4)
- Article 3: tmp_test_article3 (Page 5)

Test completed. Please check merged_articles.pdf
